# Tutorial 6: Classical Potential Relaxation

This tutorial demonstrates how to use AmorphGen's built-in classical pair potentials
(Lennard-Jones and Buckingham+Coulomb) for structure relaxation — **no GPU required**.

Classical potentials are useful for:
- Initial structure preparation before MLIP refinement
- Quick screening of large ensembles
- Systems where MLIP training data is limited
- Teaching and prototyping workflows

**Systems covered:** SiO₂ (BKS), Al₂O₃ (Catlow), TiO₂ (Matsui-Akaogi)

In [ ]:
import numpy as np
from ase.io import write
from ase.optimize import FIRE
from ase.neighborlist import neighbor_list
import matplotlib.pyplot as plt

from amorphgen.pipeline.random_gen import generate_random
from amorphgen.utils.calculators import get_calculator

---
## 4.1 Define classical potentials

Parameters are passed as a dictionary. Pair keys use tuple format `("Si", "O")`
in Python, or `"Si-O"` in YAML config.

In [ ]:
# Well-known Buckingham potentials from the literature
POTENTIALS = {
    "SiO2_BKS": {
        "composition": {"Si": 16, "O": 32},
        "classical_params": {
            "params": {
                ("Si", "O"): {"A": 18003.7572, "rho": 0.205205, "C": 133.5381},
                ("O", "O"):  {"A": 1388.7730,  "rho": 0.362319, "C": 175.0},
            },
            "charges": {"Si": 2.4, "O": -1.2},
            "cutoff": 10.0,
        },
        "ref": "BKS (van Beest et al. PRL 1990)",
    },
    "Al2O3_Catlow": {
        "composition": {"Al": 16, "O": 24},
        "classical_params": {
            "params": {
                ("Al", "O"): {"A": 12201.417, "rho": 0.195628, "C": 31.997},
                ("O", "O"):  {"A": 2029.2204, "rho": 0.343645, "C": 192.58},
            },
            "charges": {"Al": 3.0, "O": -2.0},
            "cutoff": 10.0,
        },
        "ref": "Bush et al. J. Mater. Chem. 1994",
    },
    "TiO2_MA": {
        "composition": {"Ti": 16, "O": 32},
        "classical_params": {
            "params": {
                ("Ti", "O"): {"A": 16957.53, "rho": 0.194, "C": 12.59},
                ("Ti", "Ti"): {"A": 31120.2, "rho": 0.154, "C": 5.25},
                ("O", "O"):  {"A": 11782.76, "rho": 0.234, "C": 30.22},
            },
            "charges": {"Ti": 2.196, "O": -1.098},
            "cutoff": 10.0,
        },
        "ref": "Matsui-Akaogi, Mol. Simul. 1991",
    },
}

print(f"Defined {len(POTENTIALS)} potentials")

---
## 4.2 Generate and relax structures

Use `get_calculator("buckingham", ...)` to create a classical calculator,
then relax with FIRE at `fmax=0.05` (Wolf summation limits tighter convergence).

In [ ]:
results = {}

for name, pot in POTENTIALS.items():
    print(f"\n--- {name} ({pot['ref']}) ---")
    
    # Generate random structure (auto minsep + density)
    atoms = generate_random(pot["composition"], seed=42)
    print(f"  Generated: {len(atoms)} atoms, cell={atoms.cell.lengths()[0]:.2f} A")
    
    # Create classical calculator via the unified factory
    calc = get_calculator("buckingham",
                          classical_params=pot["classical_params"])
    atoms.calc = calc
    
    e0 = atoms.get_potential_energy() / len(atoms)
    print(f"  Before opt: E/atom = {e0:.3f} eV")
    
    # Relax with FIRE (no cell filter -- classical has no stress)
    opt = FIRE(atoms, logfile=None)
    opt.run(fmax=0.05, steps=3000)
    
    ef = atoms.get_potential_energy() / len(atoms)
    fmax = np.max(np.abs(atoms.get_forces()))
    print(f"  After opt:  E/atom = {ef:.3f} eV, fmax = {fmax:.4f}, steps = {opt.nsteps}")
    
    results[name] = {"atoms": atoms, "e0": e0, "ef": ef, "steps": opt.nsteps}

---
## 4.3 Analyse coordination numbers

In [ ]:
def coordination_numbers(atoms, central, neighbour, cutoff=3.0):
    """Compute CN for central-neighbour pairs."""
    ii, jj, dd = neighbor_list("ijd", atoms, cutoff=cutoff)
    syms = atoms.get_chemical_symbols()
    cns = []
    for i in range(len(atoms)):
        if syms[i] != central:
            continue
        cn = sum(1 for idx in range(len(ii))
                 if ii[idx] == i and syms[jj[idx]] == neighbour)
        cns.append(cn)
    return np.array(cns)

# Analyse each system
print(f"{'System':<16} {'CN(M-O)':>10} {'Range':>10} {'Expected':>10}")
print("-" * 50)

expected = {"SiO2_BKS": "4.0", "Al2O3_Catlow": "4-5", "TiO2_MA": "5-6"}

for name, r in results.items():
    atoms = r["atoms"]
    # Find cation
    cation = [s for s in set(atoms.get_chemical_symbols()) if s != "O"][0]
    cns = coordination_numbers(atoms, cation, "O", cutoff=3.0)
    print(f"{name:<16} {np.mean(cns):>10.1f} {f'[{min(cns)}-{max(cns)}]':>10} {expected[name]:>10}")

---
## 4.4 Compare RDFs

In [ ]:
def compute_rdf(atoms, rmax=6.0, nbins=150):
    """Compute total radial distribution function."""
    ii, jj, dd = neighbor_list("ijd", atoms, cutoff=rmax)
    hist, edges = np.histogram(dd, bins=nbins, range=(0, rmax))
    dr = edges[1] - edges[0]
    r = edges[:-1] + dr / 2
    vol = atoms.get_volume()
    n = len(atoms)
    rho = n / vol
    norm = 4 * np.pi * r**2 * dr * rho * n
    g = hist / np.where(norm > 0, norm, 1)
    return r, g

fig, axes = plt.subplots(1, 3, figsize=(14, 4))

for ax, (name, r) in zip(axes, results.items()):
    rr, gr = compute_rdf(r["atoms"])
    ax.plot(rr, gr, "b-", lw=1.5)
    ax.set_xlabel("r (A)")
    ax.set_ylabel("g(r)")
    ax.set_title(name)
    ax.set_xlim(1, 6)

plt.tight_layout()
plt.show()

---
## 4.5 CLI equivalent

The same workflow can be run from the command line using a YAML config:

```bash
amorphgen --random-gen --composition Si=16,O=32 --relax \
    --config classical_sio2.yaml --n-structures 10
```

Where `classical_sio2.yaml` contains:

```yaml
model: buckingham
device: cpu
classical_params:
  params:
    Si-O: {A: 18003.7572, rho: 0.205205, C: 133.5381}
    O-O:  {A: 1388.7730,  rho: 0.362319, C: 175.0}
  charges: {Si: 2.4, O: -1.2}
  cutoff: 10.0
opt:
  fmax: 0.05
  optimizer: FIRE
  cell_filter: none
```

See `amorphgen/configs/example_classical.yaml` for more examples.

---
## 4.6 Hybrid workflow: Classical then MLIP

A practical workflow is to pre-relax with a classical potential (fast, no GPU),
then refine with an MLIP (accurate, GPU). This is particularly useful on HPC
where GPU time is limited.

In [ ]:
# Step 1: Classical pre-relaxation (already done above)
atoms_classical = results["SiO2_BKS"]["atoms"].copy()
print(f"Classical: E/atom = {results['SiO2_BKS']['ef']:.3f} eV")

# Step 2: Refine with MACE (uncomment to run -- requires GPU or patience)
# calc_mace = get_calculator("mace-mpa-0", device="cpu")
# atoms_classical.calc = calc_mace
# opt = FIRE(atoms_classical, logfile=None)
# opt.run(fmax=0.01, steps=500)
# print(f"MACE: E/atom = {atoms_classical.get_potential_energy()/len(atoms_classical):.3f} eV")

---
## Summary

| Feature | Classical | MLIP |
|---------|-----------|------|
| GPU required | No | Recommended |
| Install | Built-in | `pip install amorphgen[mace]` |
| Accuracy | Potential-specific | Universal |
| Speed (48 atoms) | ~70 ms/eval | ~50-200 ms/eval (GPU) |
| Best for | Pre-relaxation, screening | Production runs |
| Convergence | fmax ~0.05 (Wolf) | fmax ~0.001 |

Classical potentials are a lightweight option for initial structure preparation.
For production-quality amorphous structures, use MLIP backends (Tutorials 1-3).